# 79 — 3D Conformer Shape & Pharmacophore Descriptors

PXR has a large, flexible hydrophobic binding cavity. Shape complementarity matters.
2D fingerprints cannot capture 3D shape — but RDKit can generate conformers and
compute 3D descriptors:
- PMI ratios (principal moments of inertia) — rodlike vs disclike vs spherical
- Asphericity, eccentricity, NPR1/2 — shape anisotropy
- 3D pharmacophore fingerprints (P2D, P3D)
- WHIM (Weighted Holistic Invariant Molecular) descriptors

Combine 3D descriptors with Morgan 2D for a richer feature set.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and len(cp)>0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors3D
from rdkit.Chem import rdMolDescriptors
import concurrent.futures

def generate_conformer(smi, seed=42):
    """Generate lowest-energy conformer via ETKDG + MMFF."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) < 0:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
    except: pass
    return mol

def compute_3d_descriptors(smi):
    """Return dict of 3D shape descriptors. Returns zeros on failure."""
    defaults = {k:0.0 for k in ["pmi1","pmi2","pmi3","npr1","npr2",
                                  "asphericity","eccentricity","inertial_shape",
                                  "gyration","glob","rdf_10","rdf_20"]}
    mol = generate_conformer(smi)
    if mol is None: return defaults
    try:
        d = {}
        # PMI (principal moments of inertia)
        d["pmi1"] = float(Descriptors3D.PMI1(mol))
        d["pmi2"] = float(Descriptors3D.PMI2(mol))
        d["pmi3"] = float(Descriptors3D.PMI3(mol))
        # Normalized (shape ratios)
        d["npr1"] = float(Descriptors3D.NPR1(mol))  # rod-like: 0→1
        d["npr2"] = float(Descriptors3D.NPR2(mol))  # disc-like: 0.5→1
        # Shape descriptors
        d["asphericity"]    = float(Descriptors3D.Asphericity(mol))
        d["eccentricity"]   = float(Descriptors3D.Eccentricity(mol))
        d["inertial_shape"] = float(Descriptors3D.InertialShapeFactor(mol))
        d["gyration"]       = float(Descriptors3D.RadiusOfGyration(mol))
        d["glob"]           = float(Descriptors3D.Spherocity(mol))
        # AUTOCORR3D (first few)
        ac3d = rdMolDescriptors.CalcAUTOCORR3D(mol)
        d["rdf_10"] = float(ac3d[10]) if len(ac3d)>10 else 0.0
        d["rdf_20"] = float(ac3d[20]) if len(ac3d)>20 else 0.0
        return d
    except: return defaults

print("Generating 3D conformers for training set (this takes a few minutes)...", flush=True)


Generating 3D conformers for training set (this takes a few minutes)...


In [5]:
# Parallel conformer generation (capped workers for resource headroom)
def batch_3d(smiles_list, max_workers=4, label=""):
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(compute_3d_descriptors, s) for s in smiles_list]
        for i, f in enumerate(concurrent.futures.as_completed(futs)):
            results.append(f.result())
            if (i+1) % 500 == 0:
                print(f"  {label}: {i+1}/{len(smiles_list)}", flush=True)
    return results

desc_tr = batch_3d(tr["smiles"].tolist(), max_workers=3, label="train")
desc_te = batch_3d(te["smiles"].tolist(), max_workers=3, label="test")

keys_3d = sorted(desc_tr[0].keys())
X_3d_tr = np.array([[d[k] for k in keys_3d] for d in desc_tr], dtype=np.float32)
X_3d_te = np.array([[d[k] for k in keys_3d] for d in desc_te], dtype=np.float32)

# Replace infinities and clip outliers
X_3d_tr = np.nan_to_num(X_3d_tr, nan=0, posinf=0, neginf=0)
X_3d_te = np.nan_to_num(X_3d_te, nan=0, posinf=0, neginf=0)

# Concatenate with 2D features
X_full_tr = np.hstack([X_tr, X_3d_tr])
X_full_te  = np.hstack([X_te, X_3d_te])
print(f"\n2D+3D features: {X_full_tr.shape}")
print(f"3D descriptor stats:\n{pd.DataFrame(X_3d_tr, columns=keys_3d).describe().round(3).to_string()}")


  train: 500/4139


  train: 1000/4139


  train: 1500/4139


  train: 2000/4139


  train: 2500/4139


  train: 3000/4139


  train: 3500/4139


  train: 4000/4139


  test: 500/513



2D+3D features: (4139, 2277)
3D descriptor stats:
       asphericity  eccentricity    glob  gyration  inertial_shape    npr1    npr2    pmi1    pmi2    pmi3  rdf_10  rdf_20
count       4139.0        4139.0  4139.0    4139.0          4139.0  4139.0  4139.0  4139.0  4139.0  4139.0  4139.0  4139.0
mean           0.0           0.0     0.0       0.0             0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0
std            0.0           0.0     0.0       0.0             0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0
min            0.0           0.0     0.0       0.0             0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0
25%            0.0           0.0     0.0       0.0             0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0
50%            0.0           0.0     0.0       0.0             0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0
75%            0.0           0.0     0.0       0.0             0.0     0.0     0.0     0

In [6]:
# Scaffold CV with 2D+3D features
print("\n=== 2D+3D scaffold CV ===", flush=True)
oof = np.full(len(y_tr), np.nan)
oof_2d = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # 2D baseline
    m2 = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                   valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                   callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_2d[va_idx] = m2.predict(X_tr[va_idx])
    # 2D+3D
    m3 = lgb.train(LGBM, lgb.Dataset(X_full_tr[tr_idx], label=y_tr[tr_idx]),
                   valid_sets=[lgb.Dataset(X_full_tr[va_idx], label=y_tr[va_idx])],
                   callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof[va_idx] = m3.predict(X_full_tr[va_idx])
    r2d = rae(y_tr[va_idx], oof_2d[va_idx])
    r3d = rae(y_tr[va_idx], oof[va_idx])
    print(f"  fold {fold+1}  2D={r2d:.4f}  2D+3D={r3d:.4f}", flush=True)

m_2d  = full_metrics(y_tr, oof_2d, cliff_pairs, "2D_only")
m_3d  = full_metrics(y_tr, oof, cliff_pairs, "2D+3D_shape")
print("\n" + pd.DataFrame([m_2d,m_3d],index=["2D","2D+3D"]).round(4).to_string())

m_final = lgb.train(LGBM, lgb.Dataset(X_full_tr,label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(m_final.predict(X_full_te), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"X_tr_3d.npy", X_3d_tr)
np.save(DATA_PROCESSED/"X_te_3d.npy", X_3d_te)
np.save(DATA_PROCESSED/"oof_3d_shape.npy", oof)
np.save(DATA_PROCESSED/"te_oof_3d_shape.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"79_3d_shape_descriptors.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")



=== 2D+3D scaffold CV ===


  fold 1  2D=0.4982  2D+3D=0.4982


  fold 2  2D=0.5759  2D+3D=0.5759


  fold 3  2D=0.6021  2D+3D=0.6021


  fold 4  2D=0.5665  2D+3D=0.5665


  fold 5  2D=0.6033  2D+3D=0.6033


  [2D_only] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan
  [2D+3D_shape] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan

          RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
2D     0.5643  0.5134  0.5991    0.774    0.7268   0.5345        NaN
2D+3D  0.5643  0.5134  0.5991    0.774    0.7268   0.5345        NaN


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\79_3d_shape_descriptors.csv
Test: min=2.35 med=4.95 max=6.00
